# Spoke silver layer - stations

## Imports

In [0]:
from pyspark.sql import functions as F, Window as W

## Assigning variables

In [0]:
SOURCE = "spoke.bronze.station"
TARGET = "spoke.silver.station"
QUARANTINE = "spoke.silver.station_quarantine"

## Reading bronze source

In [0]:
df_bronze = spark.table(SOURCE)

## Deduplicating

In [0]:
window = (W.partitionBy("station_id")
          .orderBy(
              F.desc("_source_file_modification_time"),
              F.desc("_source_file")))

df_dedup = (df_bronze
            .withColumn("rank", F.row_number().over(window))
            .filter(F.col("rank") == 1)
            .drop("rank"))

## Cleaning

### Column helpers

In [0]:
def normalize_spaces(column):
    return F.trim(
        F.regexp_replace(
            column,
            r"[\s\u00A0\u2009\u200B]+",
            " "))

### Data cleaning

In [0]:
station_name_clean = normalize_spaces(F.col("station_name"))
district_clean = F.initcap(normalize_spaces(F.col("district")))
capacity_cast = F.col("capacity").try_cast("int")
latitude_cast  = F.col("latitude").try_cast("double")
longitude_cast = F.col("longitude").try_cast("double")
commissioned_on_cast = F.col("commissioned_on").try_cast("date")

In [0]:
df_clean = (df_dedup
        .withColumn("station_name", station_name_clean)
        .withColumn("district", district_clean)
        .withColumn("capacity_cast", capacity_cast)
        .withColumn("latitude_cast", latitude_cast)
        .withColumn("longitude_cast", longitude_cast)
        .withColumn("commissioned_on_cast", commissioned_on_cast))

## Data quality check + quarantine reasons

In [0]:
malformed_station_id = ~F.regexp_like(
    str=F.col("station_id"),
    regexp=F.lit(r"^[A-Z]{2}[0-9]+$"))

malformed_capacity = (F.col("capacity_cast").isNull()
                      & F.col("capacity").isNotNull())

malformed_latitude = (F.col("latitude_cast").isNull()
                      & F.col("latitude").isNotNull())

malformed_longitude = (F.col("longitude_cast").isNull()
                       & F.col("longitude").isNotNull())

malformed_commissioned_on = (F.col("commissioned_on_cast").isNull()
                             & F.col("commissioned_on").isNotNull())

quarantine_reasons = F.array_compact(
    F.array(
        F.when(malformed_station_id, F.lit("malformed_station_id")),
        F.when(malformed_capacity, F.lit("malformed_capacity")),
        F.when(malformed_latitude, F.lit("malformed_latitude")),
        F.when(malformed_longitude, F.lit("malformed_longitude")),
        F.when(malformed_commissioned_on, F.lit("malformed_commissioned_on"))))

In [0]:
df_with_quarantine = (df_clean
                      .withColumn("quarantine_reasons", quarantine_reasons))

## Prepare quarantine and silver dfs

In [0]:
df_quarantine = (df_with_quarantine
                 .filter(F.size(F.col("quarantine_reasons")) > 0)
                 .drop("capacity_cast", "latitude_cast", "longitude_cast", "commissioned_on_cast")
                 .withColumn("_quarantined_at", F.current_timestamp()))

In [0]:
df_silver = (df_with_quarantine
             .filter(F.size(F.col("quarantine_reasons")) == 0)
             .select(
                 "station_id",
                 "station_name",
                 "district",
                 "capacity_cast",
                 "latitude_cast",
                 "longitude_cast",
                 "commissioned_on_cast",
                 "status",
                 "_ingested_at",
                 "_source_file",
                 "_ingest_batch_id",
                 "_source_file_modification_time")
             .withColumnRenamed("capacity_cast", "capacity")
             .withColumnRenamed("latitude_cast", "latitude")
             .withColumnRenamed("longitude_cast", "longitude")
             .withColumnRenamed("commissioned_on_cast", "commissioned_on")
            )

## Check if all lines are taken into account

In [0]:
silver_lines = df_silver.count()
quarantine_lines = df_quarantine.count()
total_lines = silver_lines + quarantine_lines
expected_lines = df_dedup.count()

## Load to silver layer

In [0]:
(df_quarantine
 .write.mode("append")
 .format("delta")
 .saveAsTable(QUARANTINE))

(df_silver
 .write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(TARGET))